#### Imports

In [1]:
import stim
import pymatching
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree
from datetime import datetime
from pathlib import Path
import os

In [2]:
# If CUDA is available, show more details
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"Current device: {torch.cuda.current_device()}")
else:
    print("PyTorch is using CPU only")

CUDA version: 12.1
Number of GPUs: 1
GPU name: NVIDIA GeForce GTX 1080
Current device: 0


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


#### Definitions and Ground Truth

In [4]:
def surface_code_circuit(p, d): # physical error rate, distance
  return stim.Circuit.generated(
    "surface_code:rotated_memory_z",
    rounds=d,
    distance=d,
    after_clifford_depolarization=p,
    after_reset_flip_probability=p,
    before_measure_flip_probability=p,
    before_round_data_depolarization=p)

def count_logical_errors(circuit: stim.Circuit, num_shots: int) -> int:
  # Sample the circuit.
  sampler = circuit.compile_detector_sampler()
  detection_events, observable_flips = sampler.sample(num_shots, separate_observables=True)

  # Configure a decoder using the circuit.
  detector_error_model = circuit.detector_error_model(decompose_errors=True)
  matcher = pymatching.Matching.from_detector_error_model(detector_error_model)

  # Run the decoder.
  predictions = matcher.decode_batch(detection_events)

  # basically compare predictions with observable_flips (what we should have measured)

  # Count the mistakes.
  num_errors = 0
  for shot in range(num_shots):
    actual_for_shot = observable_flips[shot]
    predicted_for_shot = predictions[shot]
    if not np.array_equal(actual_for_shot, predicted_for_shot):
        num_errors += 1
  return num_errors

def ler_mwpm(p, d): # logical error rate, minimum weight perfect matching
  num_shots = 100000
  circuit = surface_code_circuit(p, d)
  num_errors = count_logical_errors(circuit, num_shots)

  return num_errors / num_shots

def plot_mwpm():
  num_shots = 100000
  for d in [3, 5, 7]:
    xs = []
    ys = []
    yerrs = []
    for noise in np.linspace(0.001, 0.008, 8):
      ler = ler_mwpm(noise, d)
      xs.append(noise)
      ys.append(ler)
      yerrs.append(np.sqrt(ler * (1 - ler) / num_shots))
  plt.loglog()
  plt.xlabel("physical error rate")
  plt.ylabel("logical error rate per shot")
  plt.legend()
  plt.show()


#### Dataset Creation


In [ ]:
class SurfaceCodeSampler:
    """
    Sampler class for surface code quantum error correction.

    This class generates surface code circuits and creates training datasets
    with configurable error rates.

    Attributes:
        d (int): Code distance
        default_p (float): Default physical error rate
        circuit (stim.Circuit): The generated surface code circuit at default_p
        device (torch.device): Device to use for tensors
    """

    def __init__(self, d: int, p: float = 0.01, device: torch.device = None):
        """
        Initialize the sampler with a surface code circuit.

        Args:
            d (int): Code distance (determines code size and rounds)
            p (float): Default physical error rate (used if not overridden)
            device (torch.device): Device for tensors (defaults to CUDA if available)
        """
        self.d = d
        self.default_p = p
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Generate and save the default circuit
        self.circuit = self._generate_circuit(p)

        # Cache circuits for different p values to avoid regenerating
        self._circuit_cache = {p: self.circuit}

        print(f"SurfaceCodeSampler initialized:")
        print(f"  Distance: {d}")
        print(f"  Default error rate: {p}")
        print(f"  Device: {self.device}")
        print(f"  Num detectors: {self.circuit.num_detectors}")

    def _generate_circuit(self, p: float) -> stim.Circuit:
        """Generate a surface code circuit with given error rate."""
        return stim.Circuit.generated(
            "surface_code:rotated_memory_z",
            rounds=self.d,
            distance=self.d,
            after_clifford_depolarization=p,
            after_reset_flip_probability=p,
            before_measure_flip_probability=p,
            before_round_data_depolarization=p
        )

    def _get_circuit(self, p: float) -> stim.Circuit:
        """Get circuit for a given p, using cache if available."""
        if p not in self._circuit_cache:
            self._circuit_cache[p] = self._generate_circuit(p)
        return self._circuit_cache[p]

    def get_circuit(self) -> stim.Circuit:
        """Return the default circuit."""
        return self.circuit

    def sample(self,
               num_samples: int,
               p_values: list[float] = None,
               p_weights: list[float] = None,
               return_p_labels: bool = False) -> tuple:
        """
        Generate training data samples with configurable error rate distribution.

        This function generates syndrome detection data and observable flip labels.
        You can specify multiple error rates with weights to control what percentage
        of the dataset uses each error rate.

        Args:
            num_samples (int): Total number of samples to generate
            p_values (list[float], optional): Array of physical error rates to use.
                                              Defaults to [self.default_p].
            p_weights (list[float], optional): Array of weights (same length as p_values).
                                               Must sum to 1.0. Determines what fraction
                                               of samples use each error rate.
                                               Defaults to [1.0] (all samples at one rate).
            return_p_labels (bool): If True, also return which p was used for each sample.

        Returns:
            tuple: (detections, labels) or (detections, labels, p_indices) if return_p_labels
                - detections: torch.Tensor [num_samples, num_detectors] syndrome measurements (-1 or +1)
                - labels: torch.Tensor [num_samples] observable flip labels (0 or 1)
                - p_indices: torch.Tensor [num_samples] index into p_values for each sample

        Examples:
            # Single error rate (uses default p)
            detections, labels = sampler.sample(10000)

            # Single custom error rate
            detections, labels = sampler.sample(10000, p_values=[0.005], p_weights=[1.0])

            # Mixed error rates: 50% at p=0.001, 30% at p=0.003, 20% at p=0.005
            detections, labels = sampler.sample(
                10000,
                p_values=[0.001, 0.003, 0.005],
                p_weights=[0.5, 0.3, 0.2]
            )
        """
        # Handle defaults
        if p_values is None:
            p_values = [self.default_p]
        if p_weights is None:
            p_weights = [1.0]

        # Validate inputs
        if len(p_values) != len(p_weights):
            raise ValueError(f"p_values and p_weights must have same length. "
                           f"Got {len(p_values)} and {len(p_weights)}")

        weight_sum = sum(p_weights)
        if not np.isclose(weight_sum, 1.0, atol=1e-6):
            raise ValueError(f"p_weights must sum to 1.0, got {weight_sum}")

        # Calculate samples per error rate
        samples_per_p = []
        remaining = num_samples
        for i, weight in enumerate(p_weights):
            if i == len(p_weights) - 1:
                # Last one gets remaining to handle rounding
                n = remaining
            else:
                n = int(round(num_samples * weight))
                remaining -= n
            samples_per_p.append(n)

        # Generate samples for each error rate
        all_detections = []
        all_labels = []
        all_p_indices = []

        for i, (p, n) in enumerate(zip(p_values, samples_per_p)):
            if n <= 0:
                continue

            circuit = self._get_circuit(p)
            sampler = circuit.compile_detector_sampler()
            detections, flips = sampler.sample(n, separate_observables=True)

            # Convert to tensors
            # Convert detections: 0 -> -1, 1 -> +1 for easier reading
            det_np = detections.astype(np.float32) * 2 - 1
            det_tensor = torch.from_numpy(det_np)
            label_tensor = torch.from_numpy(flips.astype(np.float32).flatten())

            all_detections.append(det_tensor)
            all_labels.append(label_tensor)
            all_p_indices.append(torch.full((n,), i, dtype=torch.long))

        # Concatenate all samples
        detections = torch.cat(all_detections, dim=0).to(self.device)
        labels = torch.cat(all_labels, dim=0).to(self.device)
        p_indices = torch.cat(all_p_indices, dim=0).to(self.device)

        # Shuffle the dataset so error rates are mixed
        perm = torch.randperm(num_samples, device=self.device)
        detections = detections[perm]
        labels = labels[perm]
        p_indices = p_indices[perm]

        if return_p_labels:
            return detections, labels, p_indices
        return detections, labels